In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# ===============================
# CONFIGURATION
# ===============================

SERVICE_LEVEL = 0.95
Z = norm.ppf(SERVICE_LEVEL)

# ===============================
# LOAD FILE
# ===============================

df = pd.read_excel("C:/Users/Ex0164/indent_vs_actual_wide_analysis.xlsx")

# ===============================
# IDENTIFY DAILY INDENT COLUMNS ONLY
# ===============================

indent_cols = [
    col for col in df.columns
    if "Indent" in col
    and "Total" not in col
    and "Deviation" not in col
]

print("Indent columns used:")
print(indent_cols)

# ===============================
# CALCULATE MEAN & STD ON INDENT
# ===============================

df["Mean_Indent"] = df[indent_cols].mean(axis=1)
df["Std_Indent"] = df[indent_cols].std(axis=1)

df["Std_Indent"] = df["Std_Indent"].fillna(0)

# ===============================
# CONFIDENCE INTERVAL
# ===============================

df["Indent_CI_Lower"] = df["Mean_Indent"] - Z * df["Std_Indent"]
df["Indent_CI_Upper"] = df["Mean_Indent"] + Z * df["Std_Indent"]

df["Indent_CI_Lower"] = df["Indent_CI_Lower"].apply(lambda x: max(0, x))

# ===============================
# SAVE
# ===============================

df.to_excel("indent_confidence_band_clean.xlsx", index=False)

print("Indent confidence band calculated correctly.")


Indent columns used:
['2026-02-02 Indent', '2026-02-03 Indent', '2026-02-04 Indent', '2026-02-05 Indent', '2026-02-06 Indent', '2026-02-07 Indent', '2026-02-09 Indent', '2026-02-10 Indent', '2026-02-11 Indent']
Indent confidence band calculated correctly.


In [2]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

# Load indent confidence interval file
ci_df = pd.read_excel("C:/Users/Ex0164/indent_confidence_band_clean.xlsx")

# Load updated actual file
actual_df = pd.read_excel("D:/Part Production Report/PLAN_ACTUAL.xlsx")

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB AGAINST INDENT CI
# =====================================

df["12th_Inside_Indent_CI"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper"])
)

# =====================================
# VALIDATE 13TH FEB AGAINST INDENT CI
# =====================================

df["13th_Inside_Indent_CI"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI"].mean() * 100

print("12th Feb Hit Rate (Indent CI):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent CI):", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("Indent_CI_validation_12_13.xlsx", index=False)

12th Feb Hit Rate (Indent CI): 40.91 %
13th Feb Hit Rate (Indent CI): 42.31 %


In [4]:
import pandas as pd

# =====================================
# LOAD FILES
# =====================================

ci_df = pd.read_excel("C:/Users/Ex0164/confidence_band_analysis.xlsx")
actual_df = pd.read_excel("D:/Part Production Report/PLAN_ACTUAL.xlsx")

# =====================================
# RECOMPUTE CI USING ±2σ
# =====================================

ci_df["CI_Lower_2sigma"] = ci_df["Mean_Actual"] - 2 * ci_df["Std_Actual"]
ci_df["CI_Upper_2sigma"] = ci_df["Mean_Actual"] + 2 * ci_df["Std_Actual"]

ci_df["CI_Lower_2sigma"] = ci_df["CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE 12th & 13th ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH FEB (±2σ)
# =====================================

df["12th_Inside_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH FEB (±2σ)
# =====================================

df["13th_Inside_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

# =====================================
# HIT RATE CALCULATION
# =====================================

hit_rate_12 = df["12th_Inside_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (±2σ):", round(hit_rate_13, 2), "%")

# Save validation file
df.to_excel("CI_validation_12_13_2sigma.xlsx", index=False)

12th Feb Hit Rate (±2σ): 92.31 %
13th Feb Hit Rate (±2σ): 90.21 %


In [5]:
import pandas as pd

# =====================================
# LOAD INDENT CI FILE
# =====================================

ci_df = pd.read_excel("C:/Users/Ex0164/indent_confidence_band_clean.xlsx")
actual_df = pd.read_excel("D:/Part Production Report/PLAN_ACTUAL.xlsx")

# =====================================
# COMPUTE ±2σ ON INDENT
# =====================================

ci_df["Indent_CI_Lower_2sigma"] = ci_df["Mean_Indent"] - 2 * ci_df["Std_Indent"]
ci_df["Indent_CI_Upper_2sigma"] = ci_df["Mean_Indent"] + 2 * ci_df["Std_Indent"]

ci_df["Indent_CI_Lower_2sigma"] = ci_df["Indent_CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE 12TH & 13TH ACTUAL DATA
# =====================================

cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

actual_subset = actual_df[cols_to_merge]

df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# VALIDATE 12TH (Indent ±2σ)
# =====================================

df["12th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# VALIDATE 13TH (Indent ±2σ)
# =====================================

df["13th_Inside_Indent_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["Indent_CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["Indent_CI_Upper_2sigma"])
)

# =====================================
# HIT RATE
# =====================================

hit_rate_12 = df["12th_Inside_Indent_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_Indent_CI_2sigma"].mean() * 100

print("12th Feb Hit Rate (Indent ±2σ):", round(hit_rate_12, 2), "%")
print("13th Feb Hit Rate (Indent ±2σ):", round(hit_rate_13, 2), "%")

# Save
df.to_excel("Indent_CI_2sigma_validation.xlsx", index=False)

12th Feb Hit Rate (Indent ±2σ): 44.06 %
13th Feb Hit Rate (Indent ±2σ): 44.41 %


In [3]:
import pandas as pd
import numpy as np

# =====================================
# 1. LOAD FILES
# =====================================
ci_df = pd.read_excel("C:/Users/Ex0164/confidence_band_analysis.xlsx")
actual_df = pd.read_excel("D:/Part Production Report/PLAN_ACTUAL.xlsx")

# =====================================
# 2. RECOMPUTE CI USING ±2σ (same as your original)
# =====================================
ci_df["CI_Lower_2sigma"] = ci_df["Mean_Actual"] - 2 * ci_df["Std_Actual"]
ci_df["CI_Upper_2sigma"] = ci_df["Mean_Actual"] + 2 * ci_df["Std_Actual"]
ci_df["CI_Lower_2sigma"] = ci_df["CI_Lower_2sigma"].clip(lower=0)

# =====================================
# 3. MERGE 12th & 13th ACTUAL/PLAN DATA
# =====================================
cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]
actual_subset = actual_df[cols_to_merge]
df = ci_df.merge(actual_subset, on="Material", how="left")

# =====================================
# 4. LOGIC SIMULATION & COMPARISON
# =====================================
results = []

for _, row in df.iterrows():
    material = row["Material"]
    mu = row["Mean_Actual"]
    lower = row["CI_Lower_2sigma"]
    upper = row["CI_Upper_2sigma"]
    
    if mu <= 0:
        continue
    
    # Start simulation with zero inventory
    inventory = 0.0
    
    for date_str in ["2026-02-12", "2026-02-13"]:
        plan_col = f"{date_str} Total Production Plan"
        
        # Use the plan value as "demand/forecast" for comparison
        D = row[plan_col] if pd.notna(row[plan_col]) else mu
        
        # ── Buffer rule (your logic) ──
        days_cover = inventory / mu if mu > 0 else 0.0
        if days_cover >= 3.0:
            target_buffer = 0
        else:
            target_buffer = int(np.floor(days_cover)) + 1   # 0.xx→1, 1.xx→2, 2.xx→3
        
        target_inv = target_buffer * mu
        extra = max(0.0, target_inv - inventory)
        wish = D + extra
        
        # ── Apply stability clamp ──
        P = max(lower, min(upper, wish))
        P = round(P, 0)
        
        # Reason string
        reasons = []
        if target_buffer > 0:
            reasons.append(f"Build {target_buffer}d buffer")
        if P > D:
            reasons.append(f"+{int(P - D)} extra")
        elif P < D:
            reasons.append("Draw from stock")
        if P >= upper:
            reasons.append("Upper cap")
        elif P <= lower:
            reasons.append("Lower cap")
        reason = " | ".join(reasons) or "Steady at mean"
        
        # Projected inventory using logic's production
        end_inv = max(0.0, inventory + P - D)
        
        # Differences vs your actual plan
        diff_units = P - D
        diff_pct = (diff_units / D * 100) if D > 0 else 0
        
        results.append({
            "Material": material,
            "Date": date_str,
            "Your_Plan_Qty": D,
            "Logic_Qty": P,
            "Diff_Units": diff_units,
            "Diff_Percent": round(diff_pct, 2),
            "Days_Cover_Start": round(days_cover, 2),
            "Target_Buffer": target_buffer,
            "End_Inv_Logic": round(end_inv, 1),
            "Reason": reason,
            "Plan_Inside_Band": (lower <= D <= upper)
        })
        
        # Carry forward inventory using logic's decision
        inventory = end_inv

# =====================================
# 5. CREATE REPORT
# =====================================
report = pd.DataFrame(results)

# Keep original hit rates for reference
df["12th_Inside_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["CI_Upper_2sigma"])
)
df["13th_Inside_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

hit_12 = df["12th_Inside_CI_2sigma"].mean() * 100
hit_13 = df["13th_Inside_CI_2sigma"].mean() * 100

# Summary print
print("\n" + "="*70)
print("LOGIC vs YOUR PLANS – 12 & 13 Feb 2026")
print("="*70)
print(f"Materials processed                  : {len(report['Material'].unique())}")
print(f"Original hit rate 12th Feb (±2σ)     : {round(hit_12, 2)} %")
print(f"Original hit rate 13th Feb (±2σ)     : {round(hit_13, 2)} %")
print(f"Average |difference| (units)         : {round(report['Diff_Units'].abs().mean(), 1)}")
print(f"Average difference %                 : {round(report['Diff_Percent'].abs().mean(), 2)} %")
print(f"Cases where logic differs >10%       : {(report['Diff_Percent'].abs() > 10).sum()}")
print(f"Avg ending inventory 13th (logic)    : {round(report[report['Date'] == '2026-02-13']['End_Inv_Logic'].mean(), 1)}")
print("="*70)

# Save detailed report
report.to_excel("Logic_vs_Your_Plans_12_13_Detailed.xlsx", index=False)

# Pivot version - with flattened columns
pivot = report.pivot(
    index="Material",
    columns="Date",
    values=["Logic_Qty", "Your_Plan_Qty", "Diff_Percent", "Reason", "End_Inv_Logic"]
)

# Flatten MultiIndex columns
pivot.columns = ['_'.join([str(level) for level in col if level]).strip('_') 
                 for col in pivot.columns.values]

# Save pivot
pivot.reset_index().to_excel("Logic_vs_Your_Plans_12_13_Pivot.xlsx", index=False)

print("\nFiles saved:")
print("1. Logic_vs_Your_Plans_12_13_Detailed.xlsx   → row per material + date")
print("2. Logic_vs_Your_Plans_12_13_Pivot.xlsx      → one row per material")


LOGIC vs YOUR PLANS – 12 & 13 Feb 2026
Materials processed                  : 256
Original hit rate 12th Feb (±2σ)     : 92.31 %
Original hit rate 13th Feb (±2σ)     : 90.21 %
Average |difference| (units)         : 200.0
Average difference %                 : 59.04 %
Cases where logic differs >10%       : 339
Avg ending inventory 13th (logic)    : 396.8

Files saved:
1. Logic_vs_Your_Plans_12_13_Detailed.xlsx   → row per material + date
2. Logic_vs_Your_Plans_12_13_Pivot.xlsx      → one row per material


In [5]:
import pandas as pd
import numpy as np

# =====================================
# LOAD FILES
# =====================================
ci_df = pd.read_excel("C:/Users/Ex0164/confidence_band_analysis.xlsx")
actual_df = pd.read_excel("D:/Part Production Report/PLAN_ACTUAL.xlsx")

# =====================================
# RECOMPUTE CI USING ±2σ
# =====================================
ci_df["CI_Lower_2sigma"] = ci_df["Mean_Actual"] - 2 * ci_df["Std_Actual"]
ci_df["CI_Upper_2sigma"] = ci_df["Mean_Actual"] + 2 * ci_df["Std_Actual"]
ci_df["CI_Lower_2sigma"] = ci_df["CI_Lower_2sigma"].clip(lower=0)

# =====================================
# MERGE PLAN DATA
# =====================================
cols_to_merge = [
    "Material",
    "2026-02-12 Total Production Plan",
    "2026-02-13 Total Production Plan"
]

df = ci_df.merge(actual_df[cols_to_merge], on="Material", how="left")

# =====================================
# HIT RATE VALIDATION
# =====================================
df["12th_Inside_CI_2sigma"] = (
    (df["2026-02-12 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-12 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

df["13th_Inside_CI_2sigma"] = (
    (df["2026-02-13 Total Production Plan"] >= df["CI_Lower_2sigma"]) &
    (df["2026-02-13 Total Production Plan"] <= df["CI_Upper_2sigma"])
)

hit_rate_12 = df["12th_Inside_CI_2sigma"].mean() * 100
hit_rate_13 = df["13th_Inside_CI_2sigma"].mean() * 100

print("\nHit Rates:")
print("12th:", round(hit_rate_12,2), "%")
print("13th:", round(hit_rate_13,2), "%")

# =====================================
# LOGIC SIMULATION
# =====================================
results = []

for _, row in df.iterrows():

    material = row["Material"]
    mu = row["Mean_Actual"]
    lower = row["CI_Lower_2sigma"]
    upper = row["CI_Upper_2sigma"]

    if mu <= 0:
        continue

    inventory = 0.0

    for date in ["2026-02-12", "2026-02-13"]:

        col = f"{date} Total Production Plan"
        D = row[col] if pd.notna(row[col]) else mu

        days_cover = inventory / mu

        if days_cover >= 3:
            target = 0
        else:
            target = int(np.floor(days_cover)) + 1

        target_inv = target * mu
        extra = max(0, target_inv - inventory)

        wish = D + extra
        P = max(lower, min(upper, wish))
        P = round(P,0)

        end_inv = max(0, inventory + P - D)

        diff_units = P - D
        diff_pct = (diff_units/D*100) if D>0 else 0

        reason = f"Build {target}d" if target>0 else "Steady"

        results.append({
            "Material": material,
            "Date": date,
            "Actual_Plan_Qty": D,
            "Logic_Recommended_Qty": P,
            "Diff_Units": diff_units,
            "Diff_%": round(diff_pct,2),
            "Days_Cover_Start": round(days_cover,2),
            "Target_Buffer_Days": target,
            "End_Inventory_Logic": round(end_inv,1),
            "Reason": reason
        })

        inventory = end_inv

report_df = pd.DataFrame(results)

# =====================================
# SUMMARY
# =====================================
summary = {
    "Materials": df["Material"].nunique(),
    "Avg |Diff|": round(report_df["Diff_Units"].abs().mean(),1),
    "Avg Diff %": round(report_df["Diff_%"].abs().mean(),2),
    "Cases >10%": (report_df["Diff_%"].abs()>10).sum(),
    "Avg End Inventory": round(report_df["End_Inventory_Logic"].mean(),1)
}

print("\nSUMMARY:")
for k,v in summary.items():
    print(f"{k:20} : {v}")

# =====================================
# SAVE DETAILED REPORT
# =====================================
report_df.to_excel("Logic_vs_Actual_Detailed.xlsx", index=False)

# =====================================
# PIVOT REPORT
# =====================================
pivot = report_df.pivot(
    index="Material",
    columns="Date",
    values=["Logic_Recommended_Qty","Actual_Plan_Qty","Diff_%","Reason"]
)

pivot.columns = ['_'.join(col).strip() for col in pivot.columns.values]
pivot = pivot.reset_index()

pivot.to_excel("Logic_vs_Actual_Pivot.xlsx", index=False)

print("\nFiles created:")
print("• Logic_vs_Actual_Detailed.xlsx")
print("• Logic_vs_Actual_Pivot.xlsx")


Hit Rates:
12th: 92.31 %
13th: 90.21 %

SUMMARY:
Materials            : 286
Avg |Diff|           : 200.0
Avg Diff %           : 59.04
Cases >10%           : 339
Avg End Inventory    : 322.3

Files created:
• Logic_vs_Actual_Detailed.xlsx
• Logic_vs_Actual_Pivot.xlsx
